# Medical Appointment No-Show Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `no_show`

## 2 · Project Overview

We predict whether a patient will **miss their medical appointment** (no-show) based on demographics, health conditions, SMS reminders, and scheduling lag.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given patient age, gender, health conditions, SMS reminder status, wait time, and prior no-show history, predict whether they will miss their appointment.

## 5 · Why This Project Matters

- **No-shows waste** medical resources and delay care for others.
- Predicting no-shows enables overbooking or targeted reminders.
- Healthcare systems lose billions annually to missed appointments.
- Understanding no-show drivers helps optimize scheduling.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | age, gender, scholarship, hypertension, diabetes, sms_received, wait_days, prev_no_shows |
| **Target** | no_show (0 = showed up, 1 = no-show) |
| **Class balance** | ~70/30 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "no_show"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: no_show
Save dir: E:\Github\Machine-Learning-Projects\Classification\Medical Appointment No-Show Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 medical appointments with patient and scheduling features.

In [4]:
np.random.seed(SEED)
n = 8000
age = np.random.randint(1, 90, n)
gender = np.random.choice(["M", "F"], n, p=[0.4, 0.6])
gender_num = (gender == "F").astype(int)
scholarship = np.random.binomial(1, 0.1, n)
hypertension = np.random.binomial(1, 0.2, n)
diabetes = np.random.binomial(1, 0.07, n)
sms_received = np.random.binomial(1, 0.35, n)
wait_days = np.random.poisson(10, n).clip(0, 60)
prev_no_shows = np.random.poisson(0.5, n).clip(0, 10)

score = (-0.005 * age + 0.1 * scholarship + 0.05 * wait_days / 10
         + 0.3 * prev_no_shows - 0.15 * sms_received - 0.05 * hypertension
         + np.random.normal(0, 0.8, n))
no_show = (score > np.percentile(score, 70)).astype(int)

df = pd.DataFrame({"age": age, "gender": gender, "scholarship": scholarship,
                    "hypertension": hypertension, "diabetes": diabetes,
                    "sms_received": sms_received, "wait_days": wait_days,
                    "prev_no_shows": prev_no_shows, "no_show": no_show})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['no_show'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 9)
Class balance:
no_show
0    0.7
1    0.3
Name: proportion, dtype: float64


,age,gender,scholarship,hypertension,diabetes,sms_received,wait_days,prev_no_shows,no_show
0,52,M,0,0,0,0,7,1,0
1,15,F,0,0,0,1,15,1,0
2,72,F,0,0,0,0,7,0,0
3,61,F,0,0,0,1,11,0,1
4,21,F,0,0,0,0,12,1,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 9)

Missing values:
age              0
gender           0
scholarship      0
hypertension     0
diabetes         0
sms_received     0
wait_days        0
prev_no_shows    0
no_show          0
dtype: int64

Duplicate rows: 1044

Dtypes:
age               int32
gender           object
scholarship       int32
hypertension      int32
diabetes          int32
sms_received      int32
wait_days         int32
prev_no_shows     int32
no_show           int64
dtype: object

Target 'no_show' confirmed. Value counts:
no_show
0    5600
1    2400
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["age", "wait_days", "prev_no_shows"]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(num_cols):
    df.boxplot(column=col, by=TARGET, ax=axes[i])
    axes[i].set_title(col)
plt.suptitle("Feature Distributions by No-Show Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("sms_received")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("No-Show Rate by SMS Received")
df.groupby("gender")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("No-Show Rate by Gender")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `no_show`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("No-Show Distribution")
ax.set_xticklabels(["Showed Up (0)", "No-Show (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"No-show rate: {df[TARGET].mean():.1%}")

No-show rate: 30.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 8), Test: (1600, 8)
Train class distribution:
no_show
0    0.7
1    0.3
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `gender` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~30% no-show — moderate imbalance, stratified split used.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["risk_score"] = X_train["prev_no_shows"] * (1 + X_train["wait_days"] / 30)
X_test["risk_score"] = X_test["prev_no_shows"] * (1 + X_test["wait_days"] / 30)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['age', 'gender', 'scholarship', 'hypertension', 'diabetes', 'sms_received', 'wait_days', 'prev_no_shows', 'risk_score']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7081
F1       : 0.1646

              precision    recall  f1-score   support

           0       0.71      0.97      0.82      1120
           1       0.58      0.10      0.16       480

    accuracy                           0.71      1600
   macro avg       0.65      0.53      0.49      1600
weighted avg       0.67      0.71      0.63      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.614375           0.581101  0.629027  0.625781   0.645316  0.614375    0.017418
PassiveAggressiveClassifier    0.645625           0.572470  0.579790  0.643538   0.641622  0.645625    0.046515
BernoulliNB                    0.686875           0.570387  0.619458  0.660290   0.654356  0.686875    0.017227
QuadraticDiscriminantAnalysis  0.696875           0.565030  0.616918  0.658958   0.659098  0.696875    0.090985
GaussianNB                     0.706875           0.564435  0.623555  0.659975   0.670211  0.706875    0.016010
LinearDiscriminantAnalysis     0.707500           0.535714  0.635668  0.629121   0.671126  0.707500    0.018258
BaggingClassifier              0.640625           0.534375  0.534741  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6094
F1: 0.3287


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.1550  (1.0s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[30]	valid_0's binary_logloss: 0.592192
LightGBM F1: 0.0992  (0.7s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.6094  0.3287     0.3392  0.3187
Logistic Regression    0.7081  0.1646     0.5823  0.0958
CatBoost               0.7069  0.1550     0.5733  0.0896
LightGBM               0.7050  0.0992     0.5909  0.0542

Top 3 models: ['FLAML', 'Logistic Regression', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.6094
    F1       : 0.3287
    Precision: 0.3392
    Recall   : 0.3187

  Logistic Regression:
    Accuracy : 0.7081
    F1       : 0.1646
    Precision: 0.5823
    Recall   : 0.0958

  CatBoost:
    Accuracy : 0.7069
    F1       : 0.1550
    Precision: 0.5733
    Recall   : 0.0896


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.73      0.72      1120
           1       0.34      0.32      0.33       480

    accuracy                           0.61      1600
   macro avg       0.53      0.53      0.53      1600
weighted avg       0.60      0.61      0.61      1600


Total errors: 625 / 1600 (39.1%)

Sample misclassifications:
      age  gender  scholarship  hypertension  diabetes  sms_received  wait_days  prev_no_shows  risk_score  actual  predicted  correct
4497   47     0.0            0             1         0             1         14              1    1.466667       0          1    False
1957   53     0.0            0             0         0             0         17              1    1.566667       1          0    False
3104   21     1.0            0             1         0             1         18              1    1.600000       0          1    False
836    14     1.0            

## 25 · Interpretation and Business Insight

**Key findings:**
- **Previous no-shows** is the strongest predictor of future no-shows.
- **Wait days** (longer scheduling lag) increases no-show risk.
- **SMS reminders** reduce no-show probability.
- **Age** has a slight protective effect (older patients show up more).

**Business takeaway:** Target SMS reminders at high-risk patients and reduce scheduling lag.

## 26 · Limitations

1. Synthetic data with simplified medical context.
2. No appointment type or urgency level.
3. Missing transportation and distance data.
4. No weather or day-of-week effects.
5. Binary outcome ignores cancellations vs. no-shows.

## 27 · How to Improve This Project

1. Use real hospital scheduling data.
2. Add appointment type and specialty.
3. Include transportation distance and weather.
4. Add day-of-week and time-of-day features.
5. Model cancellation separately from no-show.

## 28 · Production Considerations

- Integrate with EHR/scheduling systems.
- Output no-show probability for dynamic overbooking.
- Trigger automated SMS/call reminders for high-risk patients.
- Monitor prediction accuracy by department.
- Ensure HIPAA compliance.

## 29 · Common Mistakes

1. Treating all no-shows as equal (emergency vs. routine).
2. Not accounting for seasonal patterns.
3. Using accuracy when classes are imbalanced.
4. Ignoring the cost asymmetry (overbooking vs. empty slots).
5. Not validating on different time periods.

## 30 · Mini Challenge / Exercises

1. Remove `prev_no_shows` — how much does F1 drop?
2. Bin `wait_days` into weekly buckets and compare.
3. Try cost-sensitive learning with higher weight for no-shows.
4. Compare SMS effectiveness across age groups.
5. Plot calibration curve for the best model.

## 31 · Final Summary / Key Takeaways

1. **Prior no-show history** is the strongest predictor.
2. **SMS reminders** significantly reduce no-shows.
3. **Scheduling lag** increases no-show risk.
4. **Tree-based models** handle mixed features well.
5. **Targeted interventions** based on risk scores can improve attendance.
6. **Healthcare ML** requires careful privacy and ethical considerations.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Medical Appointment No-Show Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7069,
    "f1": 0.155,
    "precision": 0.5733,
    "recall": 0.0896
  },
  "LightGBM": {
    "accuracy": 0.705,
    "f1": 0.0992,
    "precision": 0.5909,
    "recall": 0.0542
  },
  "Logistic Regression": {
    "accuracy": 0.7081,
    "f1": 0.1646,
    "precision": 0.5823,
    "recall": 0.0958
  },
  "FLAML": {
    "accuracy": 0.6094,
    "f1": 0.3287,
    "precision": 0.3392,
    "recall": 0.3187
  }
}
